In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import importlib
import quantum_function
importlib.reload(quantum_function)

from quantum_function import *
from IPython.display import display, Markdown
from joblib import Parallel, delayed, parallel_backend

In [2]:
# --- SAFETY SWITCH: esce subito se non abiliti esplicitamente ---
RUN_THIS_CELL = True   # metti True solo quando vuoi davvero eseguire

if not RUN_THIS_CELL:
    display(Markdown("Cell locked. Set `RUN_THIS_CELL = True` to run"))
    raise SystemExit



plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    "mathtext.fontset": "cm",      # font simile a LaTeX
    "font.family": "serif",
    "font.size": 14,
    "axes.unicode_minus": False
})

N_JOBS = max(1, os.cpu_count() - 1)


In [23]:
#PARAMETERS!

gamma = 1        # Atom decay rate
T_1 = 1 / gamma  # time constant for the decay

step = 0.02
endpoint = 20 
times = np.array(np.arange(0, endpoint*T_1, step*T_1)) 

css_theta = np.pi/2  # angle for the CSS state
state = css_2(css_theta, phi=0)

my_opts_dict = {
    "keep_runs_results": False,
    #"map":"parallel",
    #"normalize_output" : False,
    #"store_final_state" : False,
    #"store_states": False,
    #"store_measurement": False,
    #"progress_bar": "Enhanced",
    "num_cpus": os.cpu_count()-1,
    }

columns_label = [
                 #"Energy",
                 "Conc",
                 "Xi2_KU",
                 #"F_phi_plus",
                 #"F_phi_minus",
                 #"F_psi_plus",
                 #"F_psi_minus",
                 "Variance_z"
            ]

label_graph_column = {
    #"Energy": r"Energy",
    "Conc": r"$\overline{\mathcal{C}}$",
    "Xi2_KU": r"$\xi^2_{KU}$",
    #"F_phi_plus": r"Fidelity $|\Phi^+\rangle$",
    #"F_phi_minus": r"Fidelity $|\Phi^-\rangle$",
    #"F_psi_plus": r"Fidelity $|\Psi^+\rangle$",
    #"F_psi_minus": r"Fidelity $|\Psi^-\rangle$",
    "Variance_z": r"$\mathrm{Var}(J_z)$"
}

my_e_ops = [
    #energy_solver, 
    concurrence_for_solver_general, 
    xi_KU_solver,
    Variance_z
    ]

EOP_REGISTRY = {
    "concurrence": concurrence_for_solver_general,   
    "spin_squeezing": xi_KU_solver,          
    "var_Jz": Variance_z, 
}

my_cfg = {
    "measurement": ["photodetection", "homodyne"],
    "input_state": [
        {"type": "css", "theta": [np.pi/2], "phi": 0.0},
    
    ],
    "phi1": [0],
    "phi2": [0],
    "eta":  [1.0, 0.9, 0.7],
    "ntraj": 10,

    "e_ops": {
        # lista “logica” (stringhe)
        "names": ["concurrence", "spin_squeezing", "var_Jz"],

        # etichette per i plot (se ti fa comodo tenerle qui)
        "labels": {
            "concurrence": r"$\mathcal{C}$",
            "spin_squeezing": r"$\xi^2$",
            "var_Jz": r"$\mathrm{Var}(J_z)$",
        }
    },
    "output": {"root": "./Revolution", "format": "npz"}
}

global_spec = {
        "step": step,
        "T1": T_1,
        "gamma": gamma,
        "endpoint": endpoint,
    }


In [36]:
import itertools

def build_css(spec):
    theta = float(spec["theta"])
    phi = float(spec.get("phi", 0.0))
    return css_2(theta, phi)

STATE_REGISTRY = {
    "css": build_css,
}

def resolve_state(state_spec, as_dm=False):
    builder = STATE_REGISTRY[state_spec["type"]]
    psi = builder(state_spec)
    return ket2dm(psi) if as_dm else psi

def resolve_eops_from_names(names):
    return [EOP_REGISTRY[n] for n in names]

def as_list(x):
    # Utility: ensure x is a list
    if isinstance(x, (list, tuple, np.ndarray)): return list(x)
    return [x]

# Utility: expand input_state specifications
def expand_input_states(specs):
    out = []
    for s in as_list(specs):
        if s["type"] == "plusplus":
            out.append({"type": "plusplus"})
        elif s["type"] == "css":
            for th in as_list(s.get("theta")):
                out.append({"type": "css", "theta": float(th)})
        else:
            raise ValueError(f"input_state non riconosciuto: {s}")
    return out

# Utility: build all run specifications
def build_run_specs(cfg):
    runs = []
    #Qui i parametri in ingresso che potrei dover iterare
    for meas, st, p1, p2, eta in itertools.product(
        as_list(cfg["measurement"]),
        expand_input_states(cfg["input_state"]),
        as_list(cfg["phi1"]),
        as_list(cfg["phi2"]),
        as_list(cfg["eta"]),
    ):
        runs.append({
            "measurement": str(meas),
            "input_state": st,
            "phi1": float(p1),
            "phi2": float(p2),
            "eta":  float(eta),
            "ntraj": int(cfg["ntraj"]),
            "e_ops_names": list(cfg["e_ops"]["names"]),
        })
    return runs


def mc_chunk(run_spec, global_spec, ntraj_chunk, seed):
    np.random.seed(seed)

    #Qui run_spec
    measurement = run_spec["measurement"]
    phi1 = run_spec["phi1"]
    phi2 = run_spec["phi2"]
    eta  = run_spec["eta"]

    #Qui global_spec
    step = global_spec["step"]
    T1 = global_spec["T1"]
    gamma = global_spec["gamma"]
    endpoint = global_spec["endpoint"]
    times = np.array(np.arange(0, endpoint*T1, step*T1))


    # risolvi e_ops da nomi (così il run_spec resta serializzabile)
    e_ops = resolve_eops_from_names(run_spec["e_ops_names"])

    # risolvi lo stato: per SME io ti consiglio density matrix
    m = measurement.lower()
    as_dm = (m in ("homodyne", "heterodyne"))
    state0 = resolve_state(run_spec["input_state"], as_dm=as_dm)

    opts_local = dict(my_opts_dict)
    opts_local["num_cpus"] = 1
    opts_local.pop("map", None)

    c_ops_unobs = collapsing_operators(gamma, phi1, phi2, 1 - eta)
    c_ops_obs   = collapsing_operators(gamma, phi1, phi2, eta)


    if m == "photodetection":
        L_unobs = qutip.liouvillian(H_free, c_ops_unobs)
        sol = mcsolve(
            L_unobs, state0, times,
            c_ops=c_ops_obs,
            e_ops=e_ops,
            ntraj=ntraj_chunk,
            options=opts_local
        )

    elif m == "homodyne":
        sol = smesolve(
            H_free, state0, times,
            c_ops=c_ops_unobs,
            sc_ops=c_ops_obs,
            heterodyne=False,
            e_ops=e_ops,
            ntraj=ntraj_chunk,
            options=opts_local
        )

    elif m == "heterodyne":
        sol = smesolve(
            H_free, state0, times,
            c_ops=c_ops_unobs,
            sc_ops=c_ops_obs,
            heterodyne=True,
            e_ops=e_ops,
            ntraj=ntraj_chunk,
            options=opts_local
        )
    else:
        raise ValueError(f"measurement non riconosciuto: {measurement}")

    expect_avg = np.array(sol.expect)
    return expect_avg, ntraj_chunk




def solver_parallel(run_spec, global_spec, seed_base=12345):
    ntraj_total = int(run_spec["ntraj"])

    chunks = np.array_split(np.arange(ntraj_total), N_JOBS)
    ntrajs = [len(c) for c in chunks if len(c) > 0]

    ss = np.random.SeedSequence(seed_base)
    seeds = [int(s.generate_state(1)[0]) for s in ss.spawn(len(ntrajs))]

    with parallel_backend("loky", inner_max_num_threads=1):
        out = Parallel(n_jobs=len(ntrajs), batch_size=1, verbose=10)(
            delayed(mc_chunk)(run_spec, global_spec, n_chunk, seed)
            for n_chunk, seed in zip(ntrajs, seeds)
        )

    num = None
    den = 0
    for expect_avg, n_chunk in out:
        num = expect_avg * n_chunk if num is None else num + expect_avg * n_chunk
        den += n_chunk

    expect_total = num / den
    return pd.DataFrame(expect_total.T, columns=columns_label)

In [39]:
my_runs = build_run_specs(my_cfg)

my_df = solver_parallel(my_runs[0], global_spec)
    
my_df

[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   1 tasks      | elapsed:    1.0s
[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed:    1.0s remaining:    2.7s
[Parallel(n_jobs=7)]: Done   3 out of   7 | elapsed:    1.0s remaining:    1.4s
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed:    1.1s remaining:    0.8s
[Parallel(n_jobs=7)]: Done   5 out of   7 | elapsed:    1.7s remaining:    0.6s
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed:    1.8s finished


,Conc,Xi2_KU,Variance_z
0,0.0,1.0,0.5
1,0.0,1.0,0.5
2,0.0,1.0,0.5
3,0.0,1.0,0.5
4,0.0,1.0,0.5
...,...,...,...
995,1.0,0.0,0.6
996,1.0,0.0,0.6
997,1.0,0.0,0.6
998,1.0,0.0,0.6


In [ ]:
save_ineff_df_npz(
    my_df,
    f"{Path(my_directory)}/phi1={phi1:.6g}_phi2={phi2:.6g}.npz",
    meta=meta
)

In [22]:
ntraj = 10000

my_phis =[
    [0,0], 
    #[0, np.pi/2], 
    #[np.pi/2,0], 
    #[np.pi/2, np.pi/2]
    ]

my_directory = f".\\Graphs\\Jz_3_Ineff_Photodetection_NPZ_pi_2"
Path(my_directory).mkdir(parents=True, exist_ok=True)

for phi1, phi2 in my_phis:
    #Setting the path to save the graphs
    phi1_dir = angle_to_path(phi1)
    phi2_dir = angle_to_path(phi2)
    ineff_df = {}

    for eta, label in zip(my_eta, my_label_eta):
        ineff_df[f"{label}"] = mc_parallel(phi1, phi2, eta, gamma, ntraj)

    meta = {
        "phi1": float(phi1),
        "phi2": float(phi2),
        "ntraj": int(ntraj),
        "etas": [float(e) for e in my_eta],
    }

    save_ineff_df_npz(
        ineff_df,
        f"{Path(my_directory)}/phi1={phi1:.6g}_phi2={phi2:.6g}.npz",
        meta=meta
    )

[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   1 tasks      | elapsed: 32.5min
[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed: 32.5min remaining: 81.4min
[Parallel(n_jobs=7)]: Done   3 out of   7 | elapsed: 32.6min remaining: 43.4min
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed: 32.6min remaining: 24.4min
[Parallel(n_jobs=7)]: Done   5 out of   7 | elapsed: 32.6min remaining: 13.0min
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed: 32.7min finished
[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   1 tasks      | elapsed: 33.2min
[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed: 33.2min remaining: 83.0min
[Parallel(n_jobs=7)]: Done   3 out of   7 | elapsed: 33.3min remaining: 44.4min
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed: 33.3min remaining: 25.0min
[Parallel(n_jobs=7)]: Done   5 out of   7 | elapsed: 33.3min remaining: 13.3min
[Parallel(n_jobs=7

In [ ]:
ineff_df, meta = load_ineff_df_from_npz(f"{Path(my_directory)}/phi1={phi1:.6g}_phi2={phi2:.6g}.npz")


#My theoretical curves
theo_curves = [np.sin(css_theta)**2 * np.exp(-(1-eta)*times)*(1-np.exp(-gamma*eta/2*times)) for eta in my_eta]

#Plotting!
for e_op in columns_label:
    #Un graph per ogni e_op, al variare di eta
    plt.figure(figsize=(12,8))
    graph_name = f"Ineff_{e_op}"

    for (label, df), eta, theory in zip(ineff_df.items(), my_eta, theo_curves):
        # x: tempo normalizzato a T1, y: media delle traiettorie
        x = times/T_1
        y = df[e_op]
        line_sim, = plt.plot(x, y, label=r"$\eta$ = " + str(eta))
        #plt.plot(times/T_1, theory, '--', color=line_sim.get_color(), alpha=0.5, linewidth=2)

    #plt.ylim(-0.01, 1.02)
    plt.xlim(0,endpoint)

    #plt.axhline(1, color="k", linewidth=1.6, linestyle="-", zorder=5)
    #plt.axhspan(1, 1.05, color="k", alpha=0.7, zorder=1, label="_nolegend_")

    plt.xlabel(r"$t/T_1$")
    plt.ylabel(label_graph_column[e_op])

    phi1_tex = angle_to_tex(phi1)
    phi2_tex = angle_to_tex(phi2)

    plt.title(r"Inefficient Photodetection: Average " + label_graph_column[e_op]+ r" over time"+ rf"$, \ \phi_1={phi1_tex}\ \ \phi_2={phi2_tex}$"
        + rf"  (n$_\mathrm{{traj}}$ = {ntraj})")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=12)

    plt.savefig(Path(my_directory) / f"{graph_name}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
"""
my_phis =[
    [0,0], 
    #[0, np.pi/2], 
    #[np.pi/2,0], 
    #[np.pi/2, np.pi/2]
    ]

ntraj = 2000

for phi_pair in my_phis:

    phi1, phi2 = phi_pair
    ineff_df = {}

    for eta,label in zip(my_eta, my_label_eta):

        my_photodetection = mcsolve(
            qutip.liouvillian(H_free, collapsing_operators(gamma, phi1, phi2, 1-eta)), 
            state, times,

            c_ops = collapsing_operators(gamma, phi1, phi2, eta),
            
            e_ops=my_e_ops,
            ntraj=ntraj,
            options=my_opts_dict
        )

        ineff_df[f"{label}"] = pd.DataFrame(np.transpose(my_photodetection.expect), columns=columns_label)

    
    #Setting the path to save the graphs
    phi1_dir = angle_to_path(phi1)
    phi2_dir = angle_to_path(phi2)
    my_directory = f".\\Graphs\\Jz_3_Ineff_Photodetection_SIM_xi_100\\phi_1={phi1_dir}_phi_2={phi2_dir}"
    Path(my_directory).mkdir(parents=True, exist_ok=True)

    #My theoretical curves
    theo_curves = [np.sin(css_theta)**2 * np.exp(-(1-eta)*times)*(1-np.exp(-gamma*eta/2*times)) for eta in my_eta]

    #Plotting!
    for e_op in columns_label:
        #Un graph per ogni e_op, al variare di eta
        plt.figure(figsize=(12,8))
        graph_name = f"Ineff_{e_op}"

        for (label, df), eta, theory in zip(ineff_df.items(), my_eta, theo_curves):
            # x: tempo normalizzato a T1, y: media delle traiettorie
            x = times/T_1
            y = df[e_op]
            line_sim, = plt.plot(x, y, label=r"$\eta$ = " + str(eta))
            #plt.plot(times/T_1, theory, '--', color=line_sim.get_color(), alpha=0.5, linewidth=2)

        #plt.ylim(-0.01, 1.02)
        plt.xlim(0,endpoint)

        #plt.axhline(1, color="k", linewidth=1.6, linestyle="-", zorder=5)
        #plt.axhspan(1, 1.05, color="k", alpha=0.7, zorder=1, label="_nolegend_")

        plt.xlabel(r"$t/T_1$")
        plt.ylabel(label_graph_column[e_op])

        phi1_tex = angle_to_tex(phi1)
        phi2_tex = angle_to_tex(phi2)

        plt.title(r"Inefficient Photodetection: Average " + label_graph_column[e_op]+ r" over time"+ rf"$, \ \phi_1={phi1_tex}\ \ \phi_2={phi2_tex}$"
            + rf"  (n$_\mathrm{{traj}}$ = {ntraj})")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=12)

        plt.savefig(Path(my_directory) / f"{graph_name}.pdf", bbox_inches="tight")
        plt.show()
"""

[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   1 tasks      | elapsed:    0.8s
[Parallel(n_jobs=7)]: Done   2 out of   7 | elapsed:    0.8s remaining:    2.1s
[Parallel(n_jobs=7)]: Done   3 out of   7 | elapsed:    0.8s remaining:    1.1s
[Parallel(n_jobs=7)]: Done   4 out of   7 | elapsed:    0.9s remaining:    0.6s
[Parallel(n_jobs=7)]: Done   5 out of   7 | elapsed:    0.9s remaining:    0.3s
[Parallel(n_jobs=7)]: Done   7 out of   7 | elapsed:    1.0s finished


ValueError: Shape of passed values is (0, 1), indices imply (0, 3)